# Comparing Activation Kernels

This experiment studies how the mathematical shape of an activation function
influences neural network learning behavior.

The network architecture and training process are kept constant while the
transition kernel used by the activation function is changed.

The experiment compares:

- Logistic
- Arctangent
- Gaussian CDF
- Gaussian RBF
- Piecewise Linear
- Polynomial

The goal is not to determine a universally best activation function, but to
observe how different mathematical transformations influence:

- convergence speed,
- final approximation error,
- learned representations,
- and optimization behavior.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nnlab.kernels import (
    LogisticKernel,
    ArctangentKernel,
    GaussianCDFKernel,
    GaussianRBFKernel,
    PiecewiseLinearKernel,
    PolynomialKernel,
)

from nnlab.activations import ParameterizedActivation

from nnlab.layers import (
    Dense,
    ActivationLayer,
)

from nnlab.models import FeedForward

from nnlab.losses import MeanSquaredError

from nnlab.optimizers import SGD

In [ ]:
x = np.linspace( -3.0, 3.0, 300, ).reshape( -1, 1, )

target = x**2

In [ ]:
kernels = {
    "Logistic": LogisticKernel(),
    "Arctangent": ArctangentKernel(),
    "Gaussian CDF": GaussianCDFKernel(),
    "Gaussian RBF": GaussianRBFKernel(),
    "Piecewise Linear": PiecewiseLinearKernel(),
    "Polynomial": PolynomialKernel(),
}

In [ ]:
kernel_x = np.linspace( -5, 5, 500, )

plt.figure(figsize=( 8, 4 ))

for name, kernel in kernels.items():

    plt.plot( kernel_x, kernel.forward(kernel_x), label=name, )

plt.title( "Transition Kernel Comparison" )
plt.xlabel( "Input" )
plt.ylabel( "Kernel Response" )
plt.legend()

plt.grid(True)

plt.show()

In [ ]:
def create_model(kernel):

    activation_1 = ActivationLayer(
        ParameterizedActivation(
            kernel=kernel,
            kernel_scale=1.0,
        )
    )

    activation_2 = ActivationLayer(
        ParameterizedActivation(
            kernel=kernel,
            kernel_scale=1.0,
        )
    )

    return FeedForward(
        layers=[
            Dense(
                input_size=1,
                output_size=16,
            ),

            activation_1,

            Dense(
                input_size=16,
                output_size=16,
            ),

            activation_2,

            Dense(
                input_size=16,
                output_size=1,
            ),
        ]
    )

In [ ]:
def train_model(
    model,
    epochs=1000,
):

    loss = MeanSquaredError()

    optimizer = SGD(
        learning_rate=0.01,
    )

    history = []

    for epoch in range(epochs):

        prediction = model.forward(x)

        error = loss.forward(
            prediction,
            target,
        )

        gradient = loss.derivative(
            prediction,
            target,
        )

        model.backward(
            gradient,
        )

        optimizer.step(
            model.parameters(),
            model.gradients(),
        )

        history.append(
            error,
        )

    return history

In [ ]:
results = {}

for name, kernel in kernels.items():

    print(
        f"Training {name}"
    )

    model = create_model(
        kernel,
    )

    history = train_model(
        model,
    )

    results[name] = {
        "model": model,
        "history": history,
    }

In [ ]:
plt.figure(figsize=(10,6))

for name, result in results.items():

    plt.plot(
        result["history"],
        label=name,
    )

plt.title(
    "Training Convergence by Activation Kernel"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "MSE"
)

plt.yscale(
    "log",
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    x,
    target,
    "k--",
    linewidth=3,
    label="Target",
)


for name, result in results.items():

    prediction = result["model"].forward(x)

    plt.plot(
        x,
        prediction,
        label=name,
    )


plt.title(
    "Final Function Approximation"
)

plt.xlabel(
    "Input"
)

plt.ylabel(
    "Output"
)

plt.legend()

plt.grid(True)

plt.show()

## Observations

The network architecture and optimization process were identical for every
experiment.

Only the transition kernel defining the activation function was changed.

Differences in learning behavior arise from properties of the activation
function itself, including:

- shape,
- smoothness,
- derivative behavior,
- saturation regions,
- and response locality.

This demonstrates the purpose of the transition kernel framework:

activation functions can be studied as parameterized mathematical objects
rather than isolated formulas.